### BeautifulSoup을 활용해 네이버 뉴스 크롤링

In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import re

# from datetime import datetime, timedelta
# import time

In [ ]:
# 설정한 갯수만큼 url 크롤링
category_num = [100, 101, 102, 103, 105]
urls_list = []
current_page = 1
total_news = 0
set_num = 2500  # 카테고리 별 원하는 크롤링 개수 

headers = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

for i in category_num:
    print(f"{i}번째 카테고리 시작")
    total_news = 0
    while total_news < set_num:
        url = "https://news.naver.com/main/main.naver?mode=LSD&mid=shm&sid1=" + str(i) + "#&date=%2000:00:00&page=" + str(current_page)
        web = requests.get(url, headers=headers).content
        source = BeautifulSoup(web, 'html.parser')
                    
        for a in source.select('a[class^="nclicks"]'):
            href = a.get('href')
            if href.startswith("https://n.news.naver.com") and href not in urls_list:
                urls_list.append(href)
                print(href)
                total_news += 1
            
        current_page += 1
    print(f"{i}번째 카테고리 완료")

    
# 크롤링한 총 url 개수
print(f"\n총 url 개수 : {len(urls_list)}")

- 원하는대로 크롤링이 안되는 문제 발생
- source 분석 -> BeautifulSoup을 사용하여 html을 끌어오면 뉴스 리스트가 누락돼서 넘어오는것을 발견

In [86]:
url = "https://news.naver.com/main/main.naver?mode=LSD&mid=shm&sid1=100#&date=%2000:00:00&page=1"
web = requests.get(url, headers=headers).content
source = BeautifulSoup(web, 'html.parser')

print(source)


<!DOCTYPE HTML>

<html lang="ko">
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta contents="always" name="referrer"/>
<meta content="600" http-equiv="refresh">
<meta content="width=1106" name="viewport">
<meta content="정치 : 네이버 뉴스" property="og:title"/>
<meta content="website" property="og:type"/>
<meta content="https://news.naver.com/main/main.naver?mode=LSD&amp;mid=shm&amp;sid1=100" property="og:url"/>
<meta content="https://ssl.pstatic.net/static.news/image/news/ogtag/navernews_800x420_20221201.png" property="og:image">
<meta content="국회, 행정, 국방, 외교 등 정치 분야 뉴스 제공" property="og:description"/>
<meta content="네이버" property="og:article:author">
<meta content="summary" name="twitter:card"/>
<meta content="정치 : 네이버 뉴스" name="twitter:title"/>
<meta content="네이버 뉴스" name="twitter:site"/>
<meta content="네이버 뉴스" name="twitter:creator"/>
<meta content="https://ssl.pstatic.net/static.news/image/news/ogtag/navernews_800x420_20221201.png" name="twitter

- 헤드라인 밑 20개 가량의 뉴스 누락

In [15]:
url = "https://news.naver.com/main/main.naver?mode=LSD&mid=shm&sid1=100#&date=%2000:00:00&page=1"
web = requests.get(url).content

source = BeautifulSoup(web, 'html.parser')

for a in source.select('a[class^="nclicks"]'):
    href = a.get('href')
    if href.startswith("https://n.news.naver.com") and href not in urls_list:
        print(href)

https://n.news.naver.com/mnews/article/003/0012346251?sid=100
https://n.news.naver.com/mnews/article/422/0000642203?sid=100
https://n.news.naver.com/mnews/article/014/0005135254?sid=100
https://n.news.naver.com/mnews/article/025/0003338485?sid=104
https://n.news.naver.com/mnews/article/001/0014476230?sid=104
https://n.news.naver.com/mnews/article/018/0005664112?sid=101
https://n.news.naver.com/mnews/article/015/0004942748?sid=102
https://n.news.naver.com/mnews/article/021/0002618786?sid=105
https://n.news.naver.com/mnews/article/079/0003858418?sid=105
https://n.news.naver.com/mnews/article/138/0002165899?sid=105
https://n.news.naver.com/mnews/article/052/0001991878?sid=102
https://n.news.naver.com/mnews/article/052/0001991937?sid=102
https://n.news.naver.com/mnews/article/025/0003338449?sid=102
https://n.news.naver.com/mnews/article/081/0003427062?sid=103
https://n.news.naver.com/mnews/article/008/0004992345?sid=105


- url에서 sid(카테고리)를 지정해주어도 BeautifulSoup을 사용하여 크롤링을 진행한다면 카테고리가 엉망으로 넘어옴
    - 확인해보니 메인 뉴스 리스트가 아닌 사이드 쪽 뉴스 리스트를 크롤링하게 됨.